# Guild Info & Quest History

Inspect guild state, look up quests by UUID, and explore quest lifecycle history.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from guildmaster_ai import GeneralAdventurer, GuildBuilder

guild = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(GeneralAdventurer, count=2)
    .with_guard()
    .build()
)

[guildmaster.adventurer.GeneralAdventurer] INFO: Granted talents: ['general']
[guildmaster.guildmaster] INFO: Registered adventurer '1525f906-53af-4014-9522-c25ea1793d77' with talents ['general']
[guildmaster.adventurer.GeneralAdventurer] INFO: Granted talents: ['general']
[guildmaster.guildmaster] INFO: Registered adventurer '5a0849a6-b198-4b09-a9d0-876b9a5599b3' with talents ['general']
[guildmaster.guild] INFO: Guard enabled: guard


## Guild snapshot before quests

In [3]:
info = guild.info
print(f"Guild ID:     {info.id}")
print(f"Adventurers:  {len(info.adventurers)}")
print(f"Guard:        {'enabled' if info.guard_enabled else 'disabled'}")
print(f"Total quests: {info.total_quests}")

for p in info.adventurers:
    print(f"\n  {p.name or p.id}")
    print(f"    Talents: {p.talents}")
    print(f"    Weapons: {p.weapons}")

Guild ID:     8b55b303-e54b-4e1b-a26c-a7f61771f163
Adventurers:  2
Guard:        enabled
Total quests: 0

  1525f906-53af-4014-9522-c25ea1793d77
    Talents: ['general']
    Weapons: []

  5a0849a6-b198-4b09-a9d0-876b9a5599b3
    Talents: ['general']
    Weapons: []


## Run some quests

In [4]:
r1 = await guild.post_quest("What is 1 + 1? Answer with just the number.")
r2 = await guild.post_quest("Capital of Japan? One word.")

print(f"Quest 1: {r1.summary[:80]}")
print(f"Quest 2: {r2.summary[:80]}")

[guildmaster.guild] INFO: === New quest request: What is 1 + 1? Answer with just the number. ===
[guildmaster.guildmaster] INFO: Assessing adventurer talents via LLM (2 adventurers)
[guildmaster.adventurer.GeneralAdventurer] INFO: Granted talents: ['general']
[guildmaster.adventurer.GeneralAdventurer] INFO: Granted talents: ['general']
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: What is 1 + 1? Answer with just the number.
[guildmaster.receptionist] INFO: Refining quest draft via LLM
[guildmaster.receptionist] INFO: Intake complete: 'Mathematical Calculation: Basic Addition'
[guildmaster.guildmaster] INFO: Checking feasibility for 'Mathematical Calculation: Basic Addition' (requires: ['general'])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=2 missing=[]
[guildmaster.guild] INFO: Quest created: 17fc7a4c ('Mathematical Calculation: Basic Addition')
[guildmaster.quest] INFO: Quest 17fc7a4c: draft -> posted (

Quest 1: 2
Quest 2: Tokyo


## Guild snapshot after quests

In [5]:
info = guild.info
print(f"Total:    {info.total_quests}")
print(f"Archived: {info.archived}")
print(f"Failed:   {info.failed}")

Total:    2
Archived: 2
Failed:   0


## List all quests

In [6]:
for q in guild.quests:
    print(f"[{q.id[:8]}] {q.title} -- {q.status.value}")

[89e24999] Identify Japan's Capital City -- archived
[17fc7a4c] Mathematical Calculation: Basic Addition -- archived


## Quest detail & history

Look up any quest by UUID to see its full lifecycle.

In [7]:
quest_id = guild.quests[0].id
q = guild.get_quest(quest_id)
r = guild.get_result(quest_id)

print(f"Title:    {q.title}")
print(f"Status:   {q.status.value}")
print(f"Rank:     {q.rank.name}")
print(f"Talents:  {q.required_talents}")
print(f"Result:   {r.summary[:100] if r else 'N/A'}")

print("\nLifecycle:")
for h in q.history:
    fr = h.payload.get("from", "")
    to = h.payload.get("to", "")
    print(f"  {fr} -> {to}  (by {h.actor})")

Title:    Identify Japan's Capital City
Status:   archived
Rank:     E
Talents:  ['general']
Result:   Tokyo

Lifecycle:
  draft -> posted  (by quest_board)
  posted -> assigned  (by guildmaster)
  assigned -> in_progress  (by 2e6f8481-5e42-4905-9578-eeceaad89aba)
  in_progress -> completed  (by guildmaster)
  completed -> archived  (by librarian)


## Quest board

The quest board tracks all posted quests.

In [8]:
board = guild.quest_board
print(f"Posted quests on board: {len(board.quests)}")
print(f"\nGuild repr: {guild!r}")

AttributeError: 'QuestBoard' object has no attribute 'quests'